In [1]:
import pandas as pd

def quality_report(df, key=None):
    n = len(df)

    # 1. Base table: one row per column
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(1),
        "unique": df.nunique(),
    })

    # 2. Ranges for numeric columns only
    num = df.select_dtypes("number")
    report["min"] = num.min()
    report["max"] = num.max()
    report["mean"] = num.mean().round(2)

    # 3. Text column that is secretly numbers
    def looks_numeric(s):
        if not pd.api.types.is_string_dtype(s):
            return False
        s = s.dropna()
        return len(s) > 0 and pd.to_numeric(s, errors="coerce").notna().mean() > 0.5
    report["text_but_numeric"] = df.apply(looks_numeric)

    # 4. Leading/trailing spaces in text
    def has_spaces(s):
        if not pd.api.types.is_string_dtype(s):
            return False
        s = s.dropna().astype(str)
        return (s != s.str.strip()).any()
    report["extra_spaces"] = df.apply(has_spaces)

    # 5. Table-level facts
    print(f"Rows: {n} | Columns: {df.shape[1]}")
    print(f"Fully duplicated rows: {df.duplicated().sum()}")
    if key:
        print(f"Duplicated '{key}' values: {df[key].duplicated().sum()}")

    return report

In [2]:
df = pd.read_csv('orders.csv')

In [3]:
quality_report(df, key="order_id")  # also checks the ID column

Rows: 6315 | Columns: 8
Fully duplicated rows: 25
Duplicated 'order_id' values: 25


,dtype,missing,missing_%,unique,min,max,mean,text_but_numeric,extra_spaces
order_id,str,0,0.0,6290,NaN,NaN,NaN,False,False
order_date,str,0,0.0,432,NaN,NaN,NaN,False,False
customer_id,str,45,0.7,394,NaN,NaN,NaN,False,False
product_id,str,0,0.0,13,NaN,NaN,NaN,False,False
quantity,int64,0,0.0,5,-1.0,4.0,1.62,False,False
discount_pct,str,0,0.0,11,NaN,NaN,NaN,True,False
status,str,0,0.0,12,NaN,NaN,NaN,False,True
payment_method,str,0,0.0,3,NaN,NaN,NaN,False,False
